# 02 Multi Image Meta Tag Tutorial

This tutorial will guide you through the key components of using the Image Meta Tag package to create an image-meta-tag webpage with multiple images. 
It is assumed you have reviewed `01_image_tag_tutorial.ipynb`

The goal is to develop the following files in your `public_html` directory:
* A database file which will contain image locations and their associated tags. 
* Symlinks to the actual image locations (as its likely you don't want these in your home directory)
* A webpage which will display: 
    * multiple images, 
    * selector dropdown menus 
    * slider to compare.

We should end up with the following files in our `public_html/project_dir`:

```text
├── webpage.html
├── databasefile.db
├── image1.png -> /data/scratch/user.name/image_meta_tag_test/image1.png
└── image2.png -> /data/scratch/user.name/image_meta_tag_test/image2.png
```

When you create the database you could do this with an existing directory of images as long as you have a method to create the tags for each one. You could do this by parsing the filename if it contained the tag data you require. Alternatively when you create the images you could either write out tags and image paths to the database, or export tag data when creating the charts. We do this in a project by creating json files for each image created. Each json file contains the image path and tags for that image. We can then read all the json files into the database. This approach means we don't need to tag the images, but can just create the database.

### Example
We are aiming to build a webpage with 2 images (or more) and a slider to slide over and compare the images. 

![Image Meta Tag Tutorial 2](images/image_meta_tag_tutorial_2.png)

### Setup
Check the `image_tag_environment.yml` file for environment requirements. 

In [3]:
# This tutorial uses a config file to set paths and avoid path exposure. 

import yaml
with open("config.yml", "r") as file:
    config = yaml.safe_load(file)

In [ ]:
import os
import datetime
import json

from pathlib import Path

# Import the ImageMetaTag package:
import ImageMetaTag as imt

# Suppress warnings for demo purposes
import warnings
warnings.filterwarnings("ignore")

### Project Paths
This time we already have the images generated, so just need to know where they are.

In [5]:
# Visualisation data:
image_dir = Path(config["image_dir_2"])
image_data_dir = image_dir / "image_data"

In [6]:
# Create database filename:

# web_dir specifies a folder in the user's public_html folder 
# The public_html folder can be viewed at https://wwwspice/~user.name/folder/
home = Path(os.getenv("HOME"))
project_name = "image_meta_tag_tutorial_2"
web_dir = home / "public_html" / project_name
web_dir.mkdir(parents=True, exist_ok=True)

# The database file should be saved in the web_dir with the images / or symlinks to images in another location. 
database_file_path = web_dir / "image_metadata_multi.db"

In [8]:
# Ensure web-dir is empty - useful if re-running the notebook.
# Uncomment the function call to run.

def clear_web_dir(web_dir: Path):
    for item in web_dir.iterdir():
        if item.is_file() or item.is_symlink():
            item.unlink()

# clear_web_dir(web_dir)

### Define selectors
We can define alternitive names for our selector headers like this:

In [9]:
# Define selector variables and their full names.
selectors = {"variable": "Variable", 
             "display_date": "Date",
             "lead_time": "Lead Time",
             "model": "Model",
             "level_type": "Level Type",
             "level": "Level",
}

# Define a list of selector names, plus the image location path 'target' in this case.
image_tag_keys = list(selectors.keys()) + ["target"]
print(image_tag_keys)

['variable', 'display_date', 'lead_time', 'model', 'level_type', 'level', 'target']


### Load tag data
For this project we are assuming we have already created the images. When we created them we created a json file for each image which contained the image meta-data. This enabled us to decouple the image-meta-tag package from our main project. Now we will collate all those json files into a single `image_data_list` which will contain the image paths and tags for each image. 

In [10]:
def load_image_data_files(image_data_dir: str, image_tag_keys: list[str]) -> list[dict]:
    image_data_list = []

    # Loop through directories in the image_data_dir looking for json files.
    for filepath in image_data_dir.rglob("*.json"):
        
        # Open the json file and load its contents.
        with open(filepath, 'r') as file:
            data = json.load(file)

            # Flatten the nested json structure
            for outer in data:
                for inner in outer:
                    for image_data in inner:

                        # Filter and return the list to the keys identified in image_tag_keys
                        filtered_data = {k: image_data[k] for k in image_tag_keys if k in image_data}
                        image_data_list.append(filtered_data)
    print(f"Loaded {len(image_data_list)} image data entries.")

    return image_data_list


# Loop through the json files produced when creating the visualisations
image_data_list = load_image_data_files(image_data_dir, image_tag_keys)


Loaded 364 image data entries.


### Create the database

In [12]:
# This function will create symlinks in our web_dir to the actual image location. 

def create_symlink(source: str, destination: Path):
    print(f"Creating symlink and DB entry for: {destination.name}")
    try:
        if destination.exists() or destination.is_symlink():
            destination.unlink()  # Remove existing file or symlink
        os.symlink(source, destination)
    except Exception as e:
        print(f"Could not link {source} -> {destination}: {e}")

In [13]:
def create_database(image_data_list: list[dict], webpage_dir: str, database_path: str):
    database_file_path = Path(database_path)

    for image_data in image_data_list:
        # Extract the path to the image file.
        image_path = image_data.get("target")
        image_filename = os.path.basename(image_path)

        # Add symlink from image location to webdir
        create_symlink(image_path, Path(webpage_dir) / image_filename)

        # Add a database entry. 
        imt.db.write_img_to_dbfile(database_file_path, image_filename, image_data)

create_database(image_data_list, web_dir, database_file_path)


Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_06_wind_speed_006.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_12_wind_speed_012.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_18_wind_speed_018.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250104_00_wind_speed_024.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_06_wind_speed_006.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_12_wind_speed_012.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_18_wind_speed_018.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250104_00_wind_speed_024.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_06_wind_speed_006.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_12_wind_speed_012.png
Creating symlink and DB entry for: era5_era5_cds_analysis_20250103_18_wind_speed_018.png
Creating symlink and 

### Build a single ImageDict of all images / tags
Before we build our multi-image ImageDict, we need to build a single ImageDict, this will serve as a template data structure for our multi-image ImageDict.

In [ ]:
# Function to build the single ImageDict, we use this in the next code cell. 

def build_image_dict(img_tags_dict, selectors, selector_animated, animation_direction=+1) -> imt.ImageDict:

    selector_keys = list(selectors.keys())                    # Variable names
    selector_level_names: list = list(selectors.values())     # Full names for dropdowns
    print(f"Selector column names: {selector_keys}")
    print(f"Animated selector: {selector_keys[selector_animated]}")

    # Now build the ImageDict    
    img_dict = None
    for img_file, img_info in img_tags_dict.items():
        tmp_dict = imt.dict_heirachy_from_list(img_info, img_file, selector_keys)
        if not img_dict:
            img_dict = imt.ImageDict(tmp_dict, 
                                 selector_animated=selector_animated,       # Which selector menu to animate.
                                 animation_direction=animation_direction,   # Animation direction
                                 level_names=selector_level_names,  
                                 )
        else:
            img_dict.append(imt.ImageDict(tmp_dict))
    return img_dict

In [ ]:
# Read the database file into paths and tags dict
database_file_path = web_dir / database_file_path
imt_db = imt.db.read(database_file_path)
img_file_paths = imt_db[0]
img_tags_dict = imt_db[1]

# Check if img_file_paths is None
if img_file_paths is not None:
    print(f"Database loaded with {len(img_file_paths)} images.")
else:
    print("Database loaded with 0 images (img_file_paths is None)")

# Create the img_dict for the webpage 
selector_animated = 2  # e.g., animate on the 3rd selector menu

# Now build the single ImageDict using our function above. 
img_dict = build_image_dict(img_tags_dict, selectors, selector_animated)

Database loaded with 280 images.
Selector column names: ['variable', 'display_date', 'lead_time', 'model', 'level_type', 'level']
Animated selector: lead_time


In [ ]:
# Let's review the ImageDict data structure:
print(img_dict)

ImageMetaTag ImageDict:
  wind_speed:
    2025-01-04 00:00:
      024:
        era5_era5_cds:
          pl:
            250:
              era5_era5_cds_analysis_20250105_00_wind_speed_024.png
        moglobal_mn:
          pl:
            250:
              moglobal_20250104_00_wind_speed_024.png
      006:
        era5_era5_cds:
          pl:
            250:
              era5_era5_cds_analysis_20250104_06_wind_speed_006.png
        moglobal_mn:
          pl:
            250:
              moglobal_20250104_00_wind_speed_006.png
      018:
        era5_era5_cds:
          pl:
            250:
              era5_era5_cds_analysis_20250104_18_wind_speed_018.png
        moglobal_mn:
          pl:
            250:
              moglobal_20250104_00_wind_speed_018.png
      012:
        era5_era5_cds:
          pl:
            250:
              era5_era5_cds_analysis_20250104_12_wind_speed_012.png
        moglobal_mn:
          pl:
            250:
              moglobal_20250104_00_win

### Create the multi image ImageDict
Now you have created the database, symlinks for images and a single ImageDict representing all images in your project, we can develop a multi screen webpage. 

* We wanted to create comparisons between different models. Such as: 
    * model x vs model y. 
    * model x vs reference model.
* We also wanted a way to define these comparisons so it would be easy to add new comparisons / new models in the future. 
* This approach enabled us to preprogram specific model comparisons into a drop down menu. 

In [27]:
# Read the database file into paths and tags dict
db_imgs, db_img_tags = imt.db.read(database_file_path)

In [ ]:
# Find the index of the selector you wish to group on.
# In our case we wish to group on the 'model' selector to show one model vs. another model.
tagorder = list(selectors.keys())
multi_depth = tagorder.index('model')

# Define the comparison pairs you want in the dropdown
model_comparisons = [
    {
        'label': 'panguweather_era5 vs moglobal',
        'left': 'panguweather_era5_era5_cds',
        'right': 'moglobal_mn'
    },
    {
        'label': 'panguweather_era5 vs era5',
        'left': 'panguweather_era5_era5_cds',
        'right': 'era5_era5_cds'
    }
]

# Create a new ImageDict for multi images based on the original ImageDict as a template.
img_dict_multi = img_dict.copy_except_dict_and_keys()
img_dict_multi.dict = {}

# Loop through the model_comparisons to create grouped images for each dropdown option
for comparison in model_comparisons:
    left_model = comparison['left']
    right_model = comparison['right']
    label = comparison['label']

    # Find all main model images in the db
    for img_file, img_info in db_img_tags.items():
        if img_info['model'] == left_model:
            
            key_lookup_left = [img_info[x] for x in tagorder]
            key_lookup_right = key_lookup_left.copy()
            key_lookup_right[multi_depth] = right_model

            left_img = img_dict.return_from_list(key_lookup_left)
            right_img = img_dict.return_from_list(key_lookup_right)

            # Only add if both images exist
            if left_img is not None and right_img is not None:
                images = [left_img, right_img]  # You can add more than 2 images using the approach above, and add them here. 
                tmp_img_info = dict(img_info)
                tmp_img_info['model'] = label  # Set dropdown label
                tmp_dict = imt.dict_heirachy_from_list(tmp_img_info, images, tagorder)
                img_dict_multi.append(imt.ImageDict(tmp_dict))

### Create header and footer html for the webpage
You can include some inline css in the `style=""`. Alternatively you can create a separate css file and supply this as a parameter to the write_full_page() function. 

In [26]:
def get_html_header_and_footer() -> tuple[str, str]:
    margin_horizontal = 5
    margin_vertical = 20
    time_now = datetime.datetime.now()

    webpage_header = f'''
    <div style="
        width: 100%;
        background: #111;
        color: #fff;
        font-weight: bold;
        font-size: 2.0em;
        display: flex;
        align-items: center;
        justify-content: space-between;
        padding:16px 24px;
        box-sizing: border-box;
        min-height: 48px;
        margin-bottom: 20px
    ">
        <span>Image Meta Tag Tutorial 2</span>
        <img src='https://www.metoffice.gov.uk//webfiles/1755514254711/images/logos/mo-green-white.svg' alt='Met Office Logo' style="height:32px; margin-left:24px;">
    </div>
    '''

    webpage_footer = f'''
    <div id='postamble' style='margin: {margin_vertical}px {margin_horizontal}px'>
        <p>Webpage generated at: {time_now.strftime('%Y-%m-%d %H:%M:%S')} by Image-Meta-Tag</p>
    </div>
    '''

    return webpage_header, webpage_footer

### Create the webpage with multiple images

In [ ]:

out_page = web_dir / f"{project_name}.html"
webpage_header, webpage_footer = get_html_header_and_footer()

imt.webpage.write_full_page(img_dict_multi, 
                            out_page,
                            project_name,
                            preamble=webpage_header,
                            postamble=webpage_footer,
                            last_img_in_list_is_slider=True,    # Add the Slider
                            last_img_still_show=True,           # Show the image on the right
                            postamble_no_imt_link=True, 
                            show_selector_names=True,           # Selector Headings
                            verbose=False, 
                            only_show_rel_url=True,
                            write_intmed_tmpfile=True,
                            show_singleton_selectors=True,  
                            )

['image_meta_tag_tutorial_2.html',
 'imt_dropdown.js',
 'img_comparison_slider_styles.css',
 'img_comparison_slider_styles.js',
 'image_meta_tag_tutorial_2.json']

Check your wwwspice/~user.name/public_html/ directory and look for your web_dir project name directory. Find the .html file in the project directory and open it! 